# 5. All-Exercises Pose Sequence Preparation

This stage replaces the squat-only engineered feature extraction step for the widening workflow.

It reads the extracted YOLO pose arrays, applies generic preprocessing and normalization, and writes a reusable pose-sequence dataset for all included exercises.

## Reason, Approach, Result Interpretation

**Reason**
- The squat-only engineered features do not transfer cleanly to the wider exercise set.
- We need a generic sequence representation that keeps the project focused on counting without locking the next stage to squat-specific assumptions.
- This stage should now rebuild the full pose-sequence dataset, including squat, so Stage 6 can compare all exercises on the same data contract.

**Approach**
- Read the full exercise pose arrays from `pose_features/*.npy` using `pose_feature_index.csv`.
- Apply confidence-aware gap filling, smoothing, torso-centered normalization, and sequence packaging.
- Write normalized pose sequences plus an index and summary CSV.

**Result interpretation**
- `status = ok` means the source pose file was converted into a normalized pose sequence.
- `valid_ratio` is the share of frames with at least half of the keypoints above the confidence threshold.
- `mean_conf` is whole-body keypoint confidence after the pose stage, not a squat-specific lower-body metric.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection')
ANNOTATION_DIR = DRIVE_PROJECT_ROOT / 'Data' / 'LLSP' / 'annotation_cleaned'
POSE_INDEX_CSV = ANNOTATION_DIR / 'pose_feature_index.csv'
POSE_SEQUENCE_DIR = ANNOTATION_DIR / 'pose_sequences'
POSE_SEQUENCE_INDEX_CSV = ANNOTATION_DIR / 'pose_sequence_index.csv'
POSE_SEQUENCE_SUMMARY_CSV = ANNOTATION_DIR / 'pose_sequence_summary.csv'

print('DRIVE_PROJECT_ROOT =', DRIVE_PROJECT_ROOT)
print('POSE_INDEX_CSV =', POSE_INDEX_CSV)
print('POSE_SEQUENCE_DIR =', POSE_SEQUENCE_DIR)
print('POSE_SEQUENCE_INDEX_CSV =', POSE_SEQUENCE_INDEX_CSV)
print('POSE_SEQUENCE_SUMMARY_CSV =', POSE_SEQUENCE_SUMMARY_CSV)

In [ ]:
!python3 $DRIVE_PROJECT_ROOT/artifacts/3_Modeling/build_pose_sequence_dataset.py \
  --index-csv $POSE_INDEX_CSV \
  --sequence-dir $POSE_SEQUENCE_DIR \
  --output-index-csv $POSE_SEQUENCE_INDEX_CSV \
  --output-summary-csv $POSE_SEQUENCE_SUMMARY_CSV \
  --overwrite

In [ ]:
import pandas as pd

summary_df = pd.read_csv(POSE_SEQUENCE_SUMMARY_CSV)
summary_df.head()

In [ ]:
print(summary_df['status'].value_counts(dropna=False))
ok_df = summary_df[summary_df['status'] == 'ok']
if len(ok_df):
    print('\nmean valid_ratio =', ok_df['valid_ratio'].mean())
    print('mean whole-body confidence =', ok_df['mean_conf'].mean())
    print('mean frames_total =', ok_df['frames_total'].mean())

In [ ]:
summary_df.groupby(['type', 'status']).size().unstack(fill_value=0)

## Output Contract for the Next Step

The next stage can now consume:

- `pose_sequence_index.csv`
- `pose_sequences/*.npy`
- `pose_sequence_summary.csv`

This keeps the project open to:
- generic pose-sequence counting models
- per-exercise counting comparisons
- later RGB or multimodal comparisons without rewriting the whole data-preparation stage